# Notebook 01: Thu thập và Khám phá Dữ liệu (Data Collection & Exploration)
**Đề tài:** Phân tích Sự cố Bất lợi Thuốc Thú y (OpenFDA Animal & Veterinary Adverse Events) năm 2025 và Dự báo Mức độ Nghiêm trọng năm 2026 (Q1)

**Môn học:** CSC17104 – Lập trình cho Khoa học Dữ liệu  
**Nhóm:** Prg4DS-G3  
- Đinh Công Khang (24120332)
- Trần Ngô Uyên Nhi (24120210)
- Huỳnh Tuấn Kiệt (24120356)

In [ ]:
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Thêm thư mục gốc vào sys.path để import các module trong src/
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.visualization import set_project_plot_style
set_project_plot_style()

---
## 1. Thu thập Dữ liệu (Data Collection — Mục 3.1)

### 1.1. Chủ đề & Bối cảnh thực tế (Subject & Real-world Context)
- **Chủ đề:** Báo cáo các biến cố/phản ứng bất lợi (Adverse Events — AE), thiếu hụt hiệu quả mong đợi (Lack of Expected Effectiveness) và lỗi chất lượng sản phẩm liên quan đến thuốc, chế phẩm sinh học và thiết bị dùng trong thú y.
- **Bối cảnh thực tế:** Giám sát cảnh giác dược thú y hậu mại (Post-market Veterinary Pharmacovigilance) giúp cơ quan quản lý (FDA CVM), bác sĩ thú y lâm sàng, hãng dược và người nuôi thú cưng/gia súc nhận diện sớm các tín hiệu rủi ro an toàn thuốc theo loài, giai đoạn tuổi, cân nặng, đường dùng và hoạt chất.

### 1.2. Tác giả, Nguồn gốc, Quần thể & Phạm vi thời gian (Author, Provenance, Population & Period)
- **Tác giả / Tổ chức thu thập & phát hành:** **U.S. Food and Drug Administration (FDA) – Center for Veterinary Medicine (CVM)** (trực thuộc Bộ Y tế và Dịch vụ Nhân sinh Hoa Kỳ — HHS). Dữ liệu xuất phát từ hệ thống cơ sở dữ liệu quốc gia **CVM Adverse Drug Experience (ADE)** và được chuẩn hóa, phân phối qua nền tảng **openFDA** (`animalandveterinary/event`).
- **Cổng tải & Manifest chính thức:**
  - Trang tải dữ liệu hàng loạt: [openFDA Animal & Veterinary Adverse Events Downloads](https://open.fda.gov/data/downloads/)
  - Manifest JSON (chứa đường dẫn phân vùng quý, `export_date`, số bản ghi): [`https://api.fda.gov/download.json`](https://api.fda.gov/download.json)
  - Đặc tả Schema & API: [openFDA Animal & Veterinary API Documentation](https://open.fda.gov/apis/animalandveterinary/event/)
- **Quần thể & Đơn vị quan sát (Population & Unit of Observation):**
  - *Quần thể mục tiêu:* Các trường hợp động vật (thú cưng như Chó, Mèo; ngựa; gia súc, gia cầm và các loài khác) sử dụng thuốc/chế phẩm thú y (chủ yếu lưu hành tại Hoa Kỳ hoặc do nhà sản xuất đăng ký với FDA báo cáo) gặp sự cố bất lợi hoặc kém hiệu quả và được gửi báo cáo về FDA CVM.
  - *Đơn vị quan sát:* Mỗi bản ghi (`unique_aer_id_number`) đại diện cho **1 báo cáo sự kiện bất lợi (Adverse Event Report)** — thường ứng với 1 cá thể bệnh (ở thú cưng) hoặc 1 đàn/nhóm vật nuôi được điều trị cùng lúc (`number_of_animals_affected > 1` ở vật nuôi trang trại).
- **Phạm vi thời gian sử dụng trong đồ án (5 phân vùng quý liên tiếp `2025q1` – `2026q1`):**
  - **Tập lịch sử & huấn luyện (2025 Q1 – Q4):** Toàn bộ 4 phân vùng quý của năm 2025 (`2025q1`–`2025q4`), dùng để phân tích chuyên sâu Q1–Q5 và huấn luyện/kiểm định chéo mô hình Học máy (Q6).
  - **Tập kiểm thử tương lai / Out-of-Time Test (2026 Q1):** Phân vùng quý đầu năm 2026 (`2026q1`), giữ độc lập để đối chiếu sự dịch chuyển xu hướng (*data/concept drift*) và đánh giá khả năng dự báo thực tế của mô hình học máy.

### 1.3. Xác minh Giấy phép & Điều khoản sử dụng (License Verification & Terms)
- **License:** **Creative Commons CC0 1.0 Universal (Public Domain Dedication)** — xác minh tại [https://open.fda.gov/license/](https://open.fda.gov/license/), trong trường `meta.license` của mỗi file JSON gốc, và toàn văn tại [https://creativecommons.org/publicdomain/zero/1.0/](https://creativecommons.org/publicdomain/zero/1.0/). Dữ liệu thuộc phạm vi công cộng, cho phép sao chép, chuẩn hóa và phân tích tự do.
- **Terms of Service & Disclaimer:** [https://open.fda.gov/terms/](https://open.fda.gov/terms/). Theo trường `meta.disclaimer` của openFDA: dữ liệu phản ánh thông tin như được báo cáo, FDA không tự động xác minh quan hệ nhân quả của từng ca riêng lẻ và dữ liệu không dùng để thay thế tư vấn y khoa/thú y lâm sàng.

### 1.4. Phương pháp thu thập & Các hạn chế/thiên lệch đã biết (Collection Method & Known Limitations/Biases)
- **Phương pháp thu thập (Giám sát Cảnh giác dược Thú y — 2 kênh thu thập):**
  1. **Báo cáo bắt buộc từ nhà sản xuất (Mandatory Reporting — 21 CFR 514.80):** Các hãng dược và nhà tài trợ thuốc thú y được cấp phép (NADA/ANADA) bắt buộc phải báo cáo các sự cố bất lợi về FDA CVM (báo cáo khẩn 15 ngày và báo cáo định kỳ) qua cổng điện tử **FDA eSubmitter** theo chuẩn bản tin **HL7 ICSR** (VICH GL24/GL42).
  2. **Báo cáo tự nguyện từ thực tế lâm sàng (Voluntary / Spontaneous Reporting):** Bác sĩ thú y (`Veterinarian`), chủ vật nuôi (`Animal Owner`) và nhân viên y tế khác gửi báo cáo tự nguyện trực tiếp qua biểu mẫu **Form FDA 1932a** trên **Safety Reporting Portal (SRP)** hoặc báo qua nhà sản xuất.
  3. **Chuẩn hóa thuật ngữ chuyên ngành:** Các triệu chứng/phản ứng được chuẩn hóa theo bộ từ điển **VeDDRA** (*Veterinary Dictionary for Drug Regulatory Activities*) và thuốc được phân loại theo hệ thống **ATCvet** (WHO).
- **6 Hạn chế & Thiên lệch cốt lõi (Limitations & Known Biases):**
  1. **Thiếu mẫu số phơi nhiễm (Lack of Denominator / No Exposure Data):** Dữ liệu chỉ gồm các ca **có báo cáo sự cố**, không có tổng số cá thể động vật đã dùng thuốc an toàn $\rightarrow$ **không thể tính tỷ lệ mắc bệnh tuyệt đối (incidence rate / absolute risk)**, chỉ kết luận trên tỷ trọng báo cáo hoặc Tỷ số chênh báo cáo (*Reporting Odds Ratio — ROR*).
  2. **Không thiết lập quan hệ nhân quả trực tiếp (No Proof of Causality):** Sự xuất hiện đồng thời của thuốc và phản ứng trong báo cáo chỉ phản ánh sự kết hợp về mặt thời gian (*temporal association*), không chứng minh thuốc là nguyên nhân gây ra phản ứng (có thể nhiễu bởi bệnh nền, tuổi già, hoặc tương tác đa thuốc).
  3. **Báo cáo thiếu & Thiên lệch mức độ nghiêm trọng (Under-reporting & Severity/Stimulated Reporting Bias):** Không phải mọi phản ứng phụ ngoài thực tế đều được báo cáo; các biến cố nặng (`serious_ae = true`, tử vong), thuốc vừa mới cấp phép (*Weber effect*) hoặc thuốc có cảnh báo truyền thông thường được báo cáo với tần suất cao hơn.
  4. **Thiên lệch theo loài và hành vi người báo cáo (Species & Reporter Bias):** Thú cưng (Chó, Mèo) được theo dõi cá thể hóa nên chiếm tỷ trọng áp đảo so với vật nuôi trang trại (vốn thường báo cáo gộp theo đàn `number_of_animals_affected > 1`). Bác sĩ thú y (`Veterinarian`), Chủ vật nuôi (`Animal Owner`) và Hãng dược cũng khác biệt rõ rệt về độ đầy đủ lâm sàng và độ trễ gửi báo cáo (`reporting_delay_days`).
  5. **Bản ghi cập nhật trùng lặp xuyên quý (Follow-up Duplicates across Quarters):** Một hồ sơ (`unique_aer_id_number`) có thể xuất hiện lại ở quý sau khi có thông tin bổ sung $\rightarrow$ cần khử trùng lặp và giữ phiên bản ở quý muộn nhất (đã cài đặt trong `src/data_loader.py`).
  6. **Dữ liệu khuyết không ngẫu nhiên & Đơn vị đo lường hỗn hợp (Missingness & Heterogeneous Units):** Định dạng JSON đa cấp (`drug[]`, `reaction[]`, `outcome[]`), nhiều trường tùy chọn bị khuyết theo cơ chế MAR/MNAR và đơn vị tuổi/cân nặng không đồng nhất (`Day/Week/Month/Year`, `lb/oz/g/kg`).

### 1.5. Công trình/Notebook công khai liên quan & Điểm khác biệt (Declare Existing Work & Differentiation)
- **Các công trình, tài liệu và repo công khai liên quan:**
  1. **Cổng khám phá tương tác chính thức của openFDA** ([openFDA Animal & Veterinary Explorer](https://open.fda.gov/apis/animalandveterinary/event/explore/)): Chỉ cung cấp thống kê đếm tần suất đơn biến (top loài, top người báo cáo, top phản ứng VeDDRA).
  2. **Kho mã nguồn hạ tầng & Công cụ truy vấn trên GitHub** ([`FDA/openfda`](https://github.com/FDA/openfda), [`OpenVet-Projects/VetClaw`](https://github.com/OpenVet-Projects/VetClaw), [`api-evangelist/openfda`](https://github.com/api-evangelist/openfda)): Tập trung vào hạ tầng nạp JSON hoặc wrapper tra cứu nhanh qua REST API.
  3. **Nghiên cứu Học máy & XAI trên dữ liệu openFDA CVM** (Sholehrasa et al., 2025 — [`arXiv:2510.01520`](https://arxiv.org/abs/2510.01520)): Kết hợp dữ liệu lịch sử 1987–2025Q1 với cấu trúc hóa học SMILES để phân loại kết cục hậu nghiệm (*Death vs. Recovery*).
  4. **Các phân tích Cảnh giác dược thú y chuyên biệt** (*Frontiers in Veterinary Science*, 2024–2025): Thường chỉ xét một thuốc đơn lẻ (như *bedinvetmab*) và so sánh tần suất thô giữa openFDA với EudraVigilance (EVVet).
- **Điểm khác biệt cốt lõi của đồ án nhóm Prg4DS-G3:**
  1. **Pipeline chuẩn hóa 4 bảng phẳng & khử trùng lặp xuyên quý:** Thiết kế `src/download_data.py` và `src/data_loader.py` có kiểm tra toàn vẹn SHA-256 và loại trùng `unique_aer_id_number` trên giai đoạn **2025 Q1–Q4 & 2026 Q1**.
  2. **Kiểm soát biến nhiễu (D4), định lượng bất định (D5) & kiểm tra độ vững (D6):** Đối chiếu *Safety Issue* vs. *Lack Of Expected Effectiveness* bằng ROR (Q3), đánh giá tương tác *Off-label $\times$ Polypharmacy* (Q4), và phân tích độ trễ báo cáo theo đối tượng gửi (Q5).
  3. **Dự báo Triage tại thời điểm tiếp nhận với kiểm thử ngoài thời gian (Out-of-Time Split):** Dự báo trực tiếp `serious_ae` (**Train 2025 Q1–Q4 $\rightarrow$ Out-of-Time Test 2026 Q1**), loại bỏ triệt để các biến rò rỉ hậu nghiệm (`outcome`, `treated_for_ae`, `duration`).

### 1.6. Lý do chọn bộ dữ liệu (Motivation)
- Độc đáo, mang tính ứng dụng liên ngành cao (Cảnh giác Dược Thú y & Khoa học Dữ liệu), quy mô lớn (hàng chục nghìn báo cáo qua 5 quý `2025q1`–`2026q1`) và cấu trúc JSON lồng nhau giúp thể hiện trọn vẹn kỹ năng làm phẳng (flatten/normalize), tiền xử lý đơn vị sinh học và thiết kế kiểm thử theo thời gian (Temporal Validation).

In [ ]:
# TODO [Giai đoạn 1]: Đọc các file JSON (2025 Q1 - 2026 Q1) từ thư mục ../data/raw/
# và tách thành 4 bảng chuẩn hóa: reports_df, drugs_df, reactions_df, outcomes_df


---
## 2. Khám phá Dữ liệu (Data Exploration — Mục 3.2)

### 2.1. Tổng quan cấu trúc & Schema (Overview)
- Kiểm tra số dòng, số cột của từng bảng sau khi flatten (`reports`, `drugs`, `reactions`, `outcomes`) qua từng quý (2025 Q1–Q4 vs. 2026 Q1).
- Ý nghĩa một dòng ở mỗi bảng và khóa liên kết `unique_aer_id_number` / `report_id`.
- Kiểm tra các bản ghi trùng lặp (`duplicated`) hoặc rỗng và nêu rõ lý do giữ/loại bỏ.

In [ ]:
# TODO [Giai đoạn 2]: Thống kê kích thước bảng, kiểu dữ liệu và kiểm tra duplicate theo unique_aer_id_number


### 2.2. Khám phá các biến số (Numerical Columns)
- Phân phối, trung tâm và độ phân tán của: tuổi vật nuôi (`animal.age`), cân nặng (`animal.weight`), số lượng cá thể bị ảnh hưởng (`number_of_animals_affected`), độ trễ báo cáo (`reporting_delay_days`), số thuốc dùng kèm (`num_drugs`).
- Phát hiện giá trị ngoại lai (IQR, Z-score) và các giá trị vô lý/placeholder (tuổi âm, cân nặng = 0 hoặc cực đại bất thường) kèm hướng xử lý.

In [ ]:
# TODO [Giai đoạn 2]: Phân tích thống kê mô tả và trực quan hóa phân phối biến số (sau khi quy đổi đơn vị)


### 2.3. Khám phá các biến phân loại (Categorical Columns)
- Kiểm tra số lượng giá trị phân biệt và tần suất của: `animal.species`, `animal.gender`, `primary_reporter`, `type_of_information`, `serious_ae`, `treated_for_ae`, `drug.route`, `drug.used_according_to_label`, `reaction.veddra_term_name`, `outcome.medical_status`.
- Phát hiện nhãn không nhất quán (in hoa/thường, từ đồng nghĩa) và gom nhóm các danh mục hiếm (`Other`).

In [ ]:
# TODO [Giai đoạn 2]: Thống kê tần suất biến phân loại và kiểm tra nhãn không đồng nhất


### 2.4. Phân tích Dữ liệu Khuyết (Missing Data Analysis)
- Số lượng và tỷ lệ % giá trị khuyết trên từng cột.
- Đánh giá cơ chế khuyết (MCAR / MAR / MNAR): ví dụ `onset_date`, `weight`, hoặc `serious_ae` bị khuyết có phụ thuộc vào `primary_reporter` hay `type_of_information` không?
- Chiến lược xử lý cho từng cột (Drop, Impute hay tách thành nhóm `'Unknown'` riêng).

In [ ]:
# TODO [Giai đoạn 2]: Bảng tổng hợp tỷ lệ missing và biểu đồ trực quan hóa mẫu khuyết


### 2.5. Mối quan hệ sơ bộ giữa các biến (Relationships)
- Tương quan giữa các biến số và bảng chéo (cross-tabulations) giữa các tổ hợp quan trọng (`species` $\times$ `serious_ae`, `type_of_information` $\times$ `primary_reporter`, xu hướng số lượng báo cáo theo quý từ 2025Q1 đến 2026).

In [ ]:
# TODO [Giai đoạn 2]: Ma trận tương quan và cross-tabulation sơ bộ


### 2.6. Tổng kết Khám phá Dữ liệu & Kế hoạch Tiền xử lý (EDA Summary & Preprocessing Plan)
1. **3–5 Quan sát cốt lõi (Key Observations):** *(Cập nhật sau khi chạy dữ liệu)*
2. **Các vấn đề chất lượng dữ liệu nghiêm trọng (Serious Data Quality Issues):** *(Đơn vị tuổi/cân nặng hỗn hợp, cấu trúc 1-nhiều lồng nhau, tỷ lệ khuyết ở các trường tùy chọn...)*
3. **Kế hoạch Tiền xử lý chuẩn hóa (Preprocessing Plan):** Lưu các bảng đã làm sạch vào `data/processed/` để sử dụng thống nhất cho Notebook 02 và Notebook 03.

In [ ]:
# TODO [Giai đoạn 2]: Lưu dữ liệu đã chuẩn hóa cơ bản vào ../data/processed/
